# Session 1: things float when the water they push aside weighs as much as they do
**A. Real-world problem.** A steel ship weighs thousands of tonnes and floats. A pebble
weighs a few grams and sinks. A block of wood floats, but sits deeper in a pond than in the
sea. One rule explains all of it: *an object pushes water out of its way, and the water
pushes back with a force equal to the weight of the water that was pushed aside.*

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
G = 9.81            # gravity, m/s^2
FRESH = 1000.0      # density of fresh water, kg/m^3
SEA = 1025.0        # density of sea water, kg/m^3 (about)

## Step 1: average density decides
Average density = total mass / total outside volume. A floating object sinks into the
water until the water it displaces has the same mass as the object. For an object with
average density `rho_obj` in a fluid of density `rho_fluid`, that means a fraction
`rho_obj / rho_fluid` of its volume is under the surface (and it sinks if that is 1 or more).

In [ ]:
def average_density(mass, volume):
    """kg/m^3 from mass (kg) and the outside volume (m^3)."""
    return mass / volume


def floats(rho_obj, rho_fluid):
    """True if the object floats (average density below the fluid's)."""
    return bool(rho_obj < rho_fluid)


def submerged_fraction(rho_obj, rho_fluid):
    """Fraction of the volume below the surface: rho_obj / rho_fluid, at most 1."""
    return float(min(1.0, rho_obj / rho_fluid))


def displaced_mass(rho_obj, volume, rho_fluid):
    """Mass (kg) of water pushed aside: equals the object's mass when it floats."""
    return rho_fluid * volume * submerged_fraction(rho_obj, rho_fluid)

## Step 2: the draft marking
For a box with a flat bottom of area `area` (m^2) floating upright, the depth below the
surface (the *draft*) is: draft = mass / (rho_fluid * area). Ships have draft marks painted
on the bow so everyone can read how deep they sit.

In [ ]:
def draft_box(mass, area, rho_fluid):
    """Depth (m) of a flat-bottomed box below the surface."""
    return mass / (rho_fluid * area)

## Step 3: three things and one trick
A wooden block, a rock and a steel boat (the same steel, hammered flat, would sink).

In [ ]:
THINGS = {
    # name: (mass kg, outside volume m^3, flat bottom area m^2)
    "wood block 0.2 m cube": (4.8, 0.008, 0.04),
    "rock 0.2 m cube": (21.6, 0.008, 0.04),
    "small steel boat 2.0 x 0.8 x 0.5 m": (150.0, 0.8, 1.6),
}
SHORT = dict(zip(THINGS, ["wood block", "rock", "steel boat"]))
STEEL = 7850.0      # density of steel, kg/m^3


def steel_volume(mass):
    """Volume (m^3) of the metal itself, if it were a solid lump."""
    return mass / STEEL

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for name, (m, v, a) in THINGS.items():
        rho = average_density(m, v)
        print(f"{name:38s} average density {rho:7.1f} kg/m^3  floats in fresh water: {floats(rho, FRESH)}"
              f"  submerged {submerged_fraction(rho, FRESH):.3f}  draft {draft_box(m, a, FRESH) if floats(rho, FRESH) else float('nan'):.3f} m")
    m_boat = THINGS["small steel boat 2.0 x 0.8 x 0.5 m"][0]
    print(f"the boat's steel as a solid lump: {steel_volume(m_boat):.4f} m^3, density {STEEL:.0f} -> sinks: {not floats(STEEL, FRESH)}")
    print(f"wood block draft: fresh {draft_box(4.8, 0.04, FRESH):.3f} m, sea {draft_box(4.8, 0.04, SEA):.3f} m")

    # figure 1: side views with the water line
    fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), sharey=True)
    for ax, (name, (m, v, a)) in zip(axs, THINGS.items()):
        rho = average_density(m, v)
        f = submerged_fraction(rho, FRESH)
        h = 1.0
        top = h * (1 - f) if f < 1 else -h
        ax.axhspan(-1.4, 0, color="tab:blue", alpha=0.25)
        ax.axhline(0, color="tab:blue")
        ax.add_patch(plt.Rectangle((-0.5, -h * f if f < 1 else -1.3), 1.0, h, color="tab:orange", alpha=0.8))
        ax.set_title(f"{name.split(' 0')[0].split(' 2')[0]}\nrho = {rho:.0f} kg/m^3, {'floats' if f < 1 else 'sinks'}", fontsize=9)
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1.4, 1.3)
        ax.set_xticks([])
    axs[0].set_ylabel("height relative to the water line (scaled)")
    fig.tight_layout()
    fig.savefig(OUT / "float_or_sink.png", dpi=120)

    # figure 2: submerged fraction versus average density
    rho = np.linspace(0, 3000, 301)
    frac = np.minimum(1.0, rho / FRESH)
    frac_sea = np.minimum(1.0, rho / SEA)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(rho, frac, label="fresh water (1000 kg/m^3)")
    ax2.plot(rho, frac_sea, "--", label="sea water (1025 kg/m^3)")
    for name, (m, v, a) in THINGS.items():
        ax2.plot(average_density(m, v), submerged_fraction(average_density(m, v), FRESH), "ko")
    ax2.set_xlabel("average density of the object (kg/m^3)")
    ax2.set_ylabel("fraction of volume under water")
    ax2.set_title("It floats while the fraction is below 1")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "fraction_vs_density.png", dpi=120)

    # figure 3: draft marks
    fig3, ax3 = plt.subplots(figsize=(5, 4))
    for extra in np.arange(0.0, 3.1, 0.5):
        d = draft_box(4.8 + extra, 0.04, FRESH)
        ax3.plot([0, 1], [-d, -d], color="tab:blue")
        ax3.text(1.03, -d, f"{4.8 + extra:.1f} kg", va="center", fontsize=8)
    ax3.set_xlim(0, 1.5)
    ax3.set_xticks([])
    ax3.set_ylabel("draft: depth below the water line (m, drawn downward)")
    ax3.set_title("Wood block: each extra kilogram sinks it 2.5 cm")
    fig3.tight_layout()
    fig3.savefig(OUT / "draft_marks.png", dpi=120)

## Where this stops
Floating is only half the story: a ship can float and still roll over. That is Session 3.

## Coding tasks
1. **Easy.** Change the masses in `lesson.py` so the wood block is 7.0 kg. Predict its draft before you run it, then check that it is still floating.
2. **Stretch.** Write `fraction_in_sea(rho_obj)` and find the density range that floats in sea water but sinks in fresh water.
3. **Explain.** Explain, using average density, why a ship that has a hole and fills with water sinks.

## Assessment
- **A. Conceptual.** Steel has a density of about 7850 kg/m^3, yet a steel ship floats. Explain, and say what would happen to the same steel hammered into a solid lump.
- **B. Numerical.** A block of mass 2.0 kg has a volume of 0.0030 m^3. Find its average density, whether it floats in fresh water, the fraction under water and the mass of water it pushes aside.
- **C. Graph and simulation interpretation.** In the graph of submerged fraction against density, the line stops rising at density 1000 (fresh water). What does an object of density 1010 kg/m^3 do in fresh water and in sea water?
- **D. Design and optimisation challenge.** Design: a flat-bottomed boat has an empty mass of 150 kg and a bottom of 2.0 m by 0.8 m. In calm fresh water it must keep a draft of no more than 0.25 m. What is the largest cargo? State your assumptions.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    wood = L.average_density(4.8, 0.008)
    checks.append((f"worked example: wood block average density is 600 kg/m^3 (got {wood:.1f})", abs(wood - 600.0) < 1e-9))
    checks.append(("worked example: the wood block floats in fresh water", L.floats(wood, L.FRESH)))
    checks.append(("worked example: 60 percent of the wood block is under water",
                   abs(L.submerged_fraction(wood, L.FRESH) - 0.6) < 1e-12))
    d = L.draft_box(4.8, 0.04, L.FRESH)
    checks.append((f"worked example: draft = 4.8 / (1000 * 0.04) = 0.120 m (got {d:.3f})", abs(d - 0.12) < 1e-12))
    checks.append(("the rock (2700 kg/m^3) sinks", not L.floats(L.average_density(21.6, 0.008), L.FRESH)))
    checks.append(("a sunk object has submerged fraction 1", L.submerged_fraction(2700.0, L.FRESH) == 1.0))
    boat = L.average_density(150.0, 0.8)
    checks.append(("the steel boat floats although steel itself sinks", L.floats(boat, L.FRESH) and not L.floats(L.STEEL, L.FRESH)))
    checks.append(("a floating object displaces water of its own mass",
                   abs(L.displaced_mass(wood, 0.008, L.FRESH) - 4.8) < 1e-9 and abs(L.displaced_mass(boat, 0.8, L.SEA) - 150.0) < 1e-9))
    checks.append(("sea water holds the block higher than fresh water", L.draft_box(4.8, 0.04, L.SEA) < L.draft_box(4.8, 0.04, L.FRESH)))
    checks.append(("exactly equal density counts as sinking (neutral), not floating", not L.floats(1000.0, 1000.0)))
    checks.append(("each extra kilogram adds 2.5 cm of draft on a 0.04 m^2 bottom",
                   abs(L.draft_box(5.8, 0.04, L.FRESH) - L.draft_box(4.8, 0.04, L.FRESH) - 0.025) < 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
